# Multi-Agent Agentic Workflow with MCP Protocol: Tool Use, Discovery, Context Sharing & Agent Registration

This notebook demonstrates an end-to-end multi-agent system that communicates via the
**Model Context Protocol (MCP)**. The system showcases four core MCP capabilities:

| Capability | Description |
|---|---|
| **Tool Calling** | Agents invoke tools through standardised MCP `tools/call` requests |
| **Dynamic Tool Discovery** | Agents query the MCP server at runtime to learn which tools are available |
| **Contextual Data Sharing & Persistence** | A shared, persistent context store lets agents read/write state across turns |
| **Agent Registration** | Agents register themselves (and their capabilities) with the MCP server on startup |

The agents use **real LLM calls** (either OpenAI or Anthropic, configurable below) and are
orchestrated through a **LangGraph** `StateGraph`.

---

## Architecture

```
                              +---------------------------------------------+
                              |              MCP Server                      |
                              |                                             |
                              |  +---------------+  +------------------+    |
                              |  | Tool Registry  |  | Agent Registry   |    |
                              |  +---------------+  +------------------+    |
                              |                                             |
                              |  +------------------+  +---------------+    |
                              |  | Context Store    |  | Request Router |    |
                              |  | (Persistent)     |  +-------+-------+    |
                              |  +------------------+          |            |
                              +--------------------------------|-----------+
                                                               |
                  +--------------------------------------------+----------------------------+
                  |                    |                        |                            |
                  v                    v                        v                            v
    +-------------------+  +-------------------+  +-------------------+  +-------------------+
    | Orchestrator Agent |  | Research Agent    |  | Analysis Agent   |  | Writer Agent      |
    | - register         |  | - register        |  | - register       |  | - register        |
    | - discover tools   |  | - call tools      |  | - call tools     |  | - call tools      |
    +--------+----------+  +--------+----------+  +--------+---------+  +--------+----------+
             |                      |                       |                     |
             |                      |                       |                     |
             +----------+-----------+-----------+-----------+                     |
                        |                                                         |
                        v                                                         v
                +-----------------+                                       +-----------------+
                | OpenAI /        |                                       | OpenAI /        |
                | Anthropic LLM   |<--------------------------------------| Anthropic LLM   |
                +-----------------+                                       +-----------------+
```

### Sequence Diagram

```
  User        Orchestrator     MCP Server      Researcher      Analyst       Writer        LLM
   |               |               |               |              |             |            |
   |               |  Phase 0: Agent Registration   |              |             |            |
   |               |----register--->|               |              |             |            |
   |               |               |<--register-----|              |             |            |
   |               |               |<-----------register-----------|             |            |
   |               |               |<--------------------register----------------|            |
   |               |               |               |              |             |            |
   |               |  Phase 1: Tool Discovery       |              |             |            |
   |               |--tools/list-->|               |              |             |            |
   |               |<--tool schemas-|               |              |             |            |
   |               |               |               |              |             |            |
   |  Phase 2: Workflow Execution  |               |              |             |            |
   |--user query-->|               |               |              |             |            |
   |               |---plan decomposition---------------------------------------->|            |
   |               |<--sub-tasks-------------------------------------------------|            |
   |               |--context/set->|               |              |             |            |
   |               |    (plan)     |               |              |             |            |
   |               |               |               |              |             |            |
   |               |---research task-------------->|              |             |            |
   |               |               |<-tools/call---|              |             |            |
   |               |               |  (web_search) |              |             |            |
   |               |               |--results----->|              |             |            |
   |               |               |               |--synthesise---------------->|            |
   |               |               |               |<-findings-------------------|            |
   |               |               |<-context/set--|              |             |            |
   |               |               | (research)    |              |             |            |
   |               |               |               |              |             |            |
   |               |---analysis task------------------------------>|             |            |
   |               |               |<--------context/get----------|             |            |
   |               |               |--------research data-------->|             |            |
   |               |               |<--------tools/call-----------|             |            |
   |               |               |          (calculator)        |             |            |
   |               |               |               |              |--analysis-->|            |
   |               |               |               |              |<-result-----|            |
   |               |               |<--------context/set----------|             |            |
   |               |               |         (analysis)           |             |            |
   |               |               |               |              |             |            |
   |               |---writing task------------------------------------------->|            |
   |               |               |<--------------------------context/get------|            |
   |               |               |---------------------------shared data----->|            |
   |               |               |               |              |             |--draft---->|
   |               |               |               |              |             |<-report----|
   |               |               |<--------------------------context/set------|            |
   |               |               |            (final_report)    |             |            |
   |               |               |               |              |             |            |
   |               |--context/get->|               |              |             |            |
   |               |<-final_report-|               |              |             |            |
   |<--response----|               |               |              |             |            |
   |               |               |               |              |             |            |
```

## 1. Install Dependencies

In [ ]:
%pip install -q langchain langchain-openai langchain-anthropic langgraph pydantic

## 2. Configuration

Set your preferred LLM provider below. The notebook supports **OpenAI** and **Anthropic**.

In [ ]:
import os

# ── Choose your LLM provider ────────────────────────────────────────────────
# Set LLM_PROVIDER to "openai" or "anthropic"
LLM_PROVIDER = os.environ.get("LLM_PROVIDER", "openai")

# API keys – replace with your own or set as environment variables
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

OPENAI_MODEL = "gpt-4o-mini"
ANTHROPIC_MODEL = "claude-sonnet-4-20250514"

print(f"LLM provider : {LLM_PROVIDER}")
print(f"Model        : {OPENAI_MODEL if LLM_PROVIDER == 'openai' else ANTHROPIC_MODEL}")

## 3. Imports

In [ ]:
from __future__ import annotations

import json
import math
import operator
import textwrap
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import (
    Annotated,
    Any,
    Callable,
    Dict,
    List,
    Literal,
    Optional,
    TypedDict,
)

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
else:
    from langchain_anthropic import ChatAnthropic

print("All imports successful.")

## 4. MCP Protocol Layer

We implement a lightweight, in-process MCP server comprising:

| Component | Responsibility |
|---|---|
| `MCPMessage` | JSON-RPC 2.0 style request / response envelope |
| `MCPToolRegistry` | Stores tool definitions; supports dynamic registration and listing |
| `MCPAgentRegistry` | Tracks registered agents and their declared capabilities |
| `MCPContextStore` | Persistent key-value store for cross-agent data sharing |
| `MCPServer` | Facade that routes incoming requests to the correct subsystem |

### 4.1 MCP Message Envelope

In [ ]:
@dataclass
class MCPMessage:
    """JSON-RPC 2.0 inspired message used for all MCP communication."""

    method: str
    params: Dict[str, Any] = field(default_factory=dict)
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    source: str = ""
    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )


@dataclass
class MCPResponse:
    """Response to an MCPMessage."""

    id: str
    result: Any = None
    error: Optional[str] = None
    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )

    @property
    def ok(self) -> bool:
        return self.error is None


print("MCPMessage / MCPResponse defined.")

### 4.2 Tool Registry (Dynamic Tool Discovery & Calling)

In [ ]:
@dataclass
class ToolDefinition:
    """Schema describing a single tool available through MCP."""

    name: str
    description: str
    parameters: Dict[str, Any]
    handler: Callable[..., Any]
    tags: List[str] = field(default_factory=list)

    def schema_dict(self) -> Dict[str, Any]:
        return {
            "name": self.name,
            "description": self.description,
            "parameters": self.parameters,
            "tags": self.tags,
        }


class MCPToolRegistry:
    """Central registry of tools that agents can discover and invoke at runtime."""

    def __init__(self) -> None:
        self._tools: Dict[str, ToolDefinition] = {}

    def register(self, tool: ToolDefinition) -> None:
        self._tools[tool.name] = tool
        print(f"  [ToolRegistry] Registered tool: {tool.name}")

    def unregister(self, name: str) -> bool:
        removed = self._tools.pop(name, None)
        if removed:
            print(f"  [ToolRegistry] Unregistered tool: {name}")
        return removed is not None

    def list_tools(self, tag_filter: Optional[str] = None) -> List[Dict[str, Any]]:
        tools = self._tools.values()
        if tag_filter:
            tools = [t for t in tools if tag_filter in t.tags]
        return [t.schema_dict() for t in tools]

    def call(self, name: str, arguments: Dict[str, Any]) -> Any:
        if name not in self._tools:
            raise KeyError(f"Tool '{name}' not found in registry")
        return self._tools[name].handler(**arguments)


print("MCPToolRegistry defined.")

### 4.3 Agent Registry (Agent Registration)

In [ ]:
@dataclass
class AgentDescriptor:
    """Metadata that an agent supplies when registering with the MCP server."""

    agent_id: str
    name: str
    description: str
    capabilities: List[str]
    registered_at: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    status: str = "active"


class MCPAgentRegistry:
    """Tracks all agents that have registered with the MCP server."""

    def __init__(self) -> None:
        self._agents: Dict[str, AgentDescriptor] = {}

    def register(self, descriptor: AgentDescriptor) -> None:
        self._agents[descriptor.agent_id] = descriptor
        print(
            f"  [AgentRegistry] Registered agent: {descriptor.name} "
            f"(id={descriptor.agent_id}, capabilities={descriptor.capabilities})"
        )

    def deregister(self, agent_id: str) -> bool:
        removed = self._agents.pop(agent_id, None)
        if removed:
            print(f"  [AgentRegistry] Deregistered agent: {removed.name}")
        return removed is not None

    def list_agents(self) -> List[Dict[str, Any]]:
        return [
            {
                "agent_id": a.agent_id,
                "name": a.name,
                "description": a.description,
                "capabilities": a.capabilities,
                "status": a.status,
            }
            for a in self._agents.values()
        ]

    def get(self, agent_id: str) -> Optional[AgentDescriptor]:
        return self._agents.get(agent_id)

    def find_by_capability(self, capability: str) -> List[AgentDescriptor]:
        return [
            a for a in self._agents.values() if capability in a.capabilities
        ]


print("MCPAgentRegistry defined.")

### 4.4 Context Store (Contextual Data Sharing & Persistence)

In [ ]:
class MCPContextStore:
    """Persistent key-value store that allows agents to share and retrieve data.

    Data is held in memory and optionally flushed to a JSON file so it
    survives across sessions.
    """

    def __init__(self, persist_path: Optional[str] = None) -> None:
        self._store: Dict[str, Any] = {}
        self._history: List[Dict[str, Any]] = []
        self._persist_path = persist_path
        if persist_path and Path(persist_path).exists():
            with open(persist_path) as f:
                saved = json.load(f)
            self._store = saved.get("store", {})
            self._history = saved.get("history", [])
            print(f"  [ContextStore] Restored {len(self._store)} keys from {persist_path}")

    def set(self, key: str, value: Any, source: str = "") -> None:
        self._store[key] = value
        entry = {
            "action": "set",
            "key": key,
            "source": source,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }
        self._history.append(entry)
        self._flush()

    def get(self, key: str, default: Any = None) -> Any:
        return self._store.get(key, default)

    def keys(self) -> List[str]:
        return list(self._store.keys())

    def get_history(self, key: Optional[str] = None) -> List[Dict[str, Any]]:
        if key is None:
            return list(self._history)
        return [h for h in self._history if h["key"] == key]

    def snapshot(self) -> Dict[str, Any]:
        return dict(self._store)

    def _flush(self) -> None:
        if not self._persist_path:
            return
        Path(self._persist_path).parent.mkdir(parents=True, exist_ok=True)
        with open(self._persist_path, "w") as f:
            json.dump({"store": self._store, "history": self._history}, f, indent=2, default=str)


print("MCPContextStore defined.")

### 4.5 MCP Server (Request Router)

In [ ]:
class MCPServer:
    """Lightweight in-process MCP server.

    Routes JSON-RPC style requests to the appropriate subsystem:
      - tools/register, tools/list, tools/call
      - agents/register, agents/list, agents/find
      - context/set, context/get, context/keys, context/history
    """

    def __init__(self, persist_path: Optional[str] = None) -> None:
        self.tool_registry = MCPToolRegistry()
        self.agent_registry = MCPAgentRegistry()
        self.context_store = MCPContextStore(persist_path=persist_path)
        self._request_log: List[Dict[str, Any]] = []

    def handle(self, msg: MCPMessage) -> MCPResponse:
        self._request_log.append(
            {"id": msg.id, "method": msg.method, "source": msg.source, "ts": msg.timestamp}
        )
        try:
            result = self._dispatch(msg)
            return MCPResponse(id=msg.id, result=result)
        except Exception as exc:
            return MCPResponse(id=msg.id, error=str(exc))

    def _dispatch(self, msg: MCPMessage) -> Any:
        method = msg.method
        p = msg.params

        # ── Tool operations ──────────────────────────────────────────
        if method == "tools/register":
            tool_def = ToolDefinition(**p)
            self.tool_registry.register(tool_def)
            return {"registered": tool_def.name}

        if method == "tools/list":
            return self.tool_registry.list_tools(tag_filter=p.get("tag"))

        if method == "tools/call":
            return self.tool_registry.call(p["name"], p.get("arguments", {}))

        # ── Agent operations ─────────────────────────────────────────
        if method == "agents/register":
            desc = AgentDescriptor(**p)
            self.agent_registry.register(desc)
            return {"registered": desc.agent_id}

        if method == "agents/list":
            return self.agent_registry.list_agents()

        if method == "agents/find":
            matches = self.agent_registry.find_by_capability(p["capability"])
            return [{"agent_id": a.agent_id, "name": a.name} for a in matches]

        # ── Context operations ───────────────────────────────────────
        if method == "context/set":
            self.context_store.set(p["key"], p["value"], source=msg.source)
            return {"stored": p["key"]}

        if method == "context/get":
            return self.context_store.get(p["key"], p.get("default"))

        if method == "context/keys":
            return self.context_store.keys()

        if method == "context/history":
            return self.context_store.get_history(key=p.get("key"))

        raise ValueError(f"Unknown MCP method: {method}")

    def get_request_log(self) -> List[Dict[str, Any]]:
        return list(self._request_log)


print("MCPServer defined.")

## 5. Define Concrete Tools

These are the tools that agents will discover and call via MCP.

In [ ]:
# ── Tool implementations ────────────────────────────────────────────────────

def web_search(query: str, max_results: int = 3) -> str:
    """Simulated web search that returns plausible results.

    Replace the body of this function with a real search client
    (e.g. Tavily, DuckDuckGo, SerpAPI) for production use.
    """
    results = [
        {
            "title": f"Result {i+1} for: {query}",
            "snippet": f"This is a simulated search snippet #{i+1} about '{query}'. "
                       f"It contains relevant information that an analyst could use.",
            "url": f"https://example.com/search?q={query.replace(' ', '+')}&p={i+1}",
        }
        for i in range(min(max_results, 5))
    ]
    return json.dumps(results, indent=2)


def calculator(expression: str) -> str:
    """Evaluates a mathematical expression safely."""
    allowed_names = {
        k: v for k, v in math.__dict__.items() if not k.startswith("_")
    }
    allowed_names["abs"] = abs
    allowed_names["round"] = round
    try:
        result = eval(expression, {"__builtins__": {}}, allowed_names)  # noqa: S307
        return json.dumps({"expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"expression": expression, "error": str(e)})


def text_summariser(text: str, max_sentences: int = 3) -> str:
    """Extracts the first N sentences as a naive summary."""
    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
    summary = ". ".join(sentences[:max_sentences]) + "."
    return json.dumps({"summary": summary, "sentence_count": min(len(sentences), max_sentences)})


def format_report(title: str, sections: List[str]) -> str:
    """Formats sections into a simple markdown report."""
    lines = [f"# {title}", ""]
    for i, section in enumerate(sections, 1):
        lines.append(f"## Section {i}")
        lines.append(section)
        lines.append("")
    return "\n".join(lines)


print("Tool functions defined.")

## 6. Instantiate the MCP Server & Register Tools

In [ ]:
CONTEXT_PERSIST_PATH = "_mcp_context_store.json"

mcp_server = MCPServer(persist_path=CONTEXT_PERSIST_PATH)

# Register tools via MCP messages (demonstrates tools/register)
tool_definitions = [
    ToolDefinition(
        name="web_search",
        description="Search the web for information on a given query.",
        parameters={
            "query": {"type": "string", "description": "The search query"},
            "max_results": {"type": "integer", "description": "Max results to return", "default": 3},
        },
        handler=web_search,
        tags=["search", "research"],
    ),
    ToolDefinition(
        name="calculator",
        description="Evaluate a mathematical expression. Supports standard math functions.",
        parameters={
            "expression": {"type": "string", "description": "Math expression to evaluate"},
        },
        handler=calculator,
        tags=["math", "analysis"],
    ),
    ToolDefinition(
        name="text_summariser",
        description="Produce a short summary from a longer text.",
        parameters={
            "text": {"type": "string", "description": "Text to summarise"},
            "max_sentences": {"type": "integer", "description": "Max sentences", "default": 3},
        },
        handler=text_summariser,
        tags=["text", "analysis"],
    ),
    ToolDefinition(
        name="format_report",
        description="Format a list of text sections into a structured markdown report.",
        parameters={
            "title": {"type": "string", "description": "Report title"},
            "sections": {"type": "array", "description": "List of section contents"},
        },
        handler=format_report,
        tags=["text", "writing"],
    ),
]

for td in tool_definitions:
    mcp_server.tool_registry.register(td)

print(f"\nTotal tools registered: {len(mcp_server.tool_registry.list_tools())}")

## 7. MCP-Aware Agent Base Class

Every agent wraps an LLM and communicates exclusively through the MCP server.

In [ ]:
def _build_llm(temperature: float = 0.3):
    """Factory that returns the configured LLM."""
    if LLM_PROVIDER == "openai":
        return ChatOpenAI(model=OPENAI_MODEL, temperature=temperature)
    return ChatAnthropic(model=ANTHROPIC_MODEL, temperature=temperature)


class MCPAgent:
    """Base class for an LLM-powered agent that uses MCP for all I/O."""

    def __init__(
        self,
        agent_id: str,
        name: str,
        description: str,
        capabilities: List[str],
        system_prompt: str,
        server: MCPServer,
        temperature: float = 0.3,
    ) -> None:
        self.agent_id = agent_id
        self.name = name
        self.server = server
        self.system_prompt = system_prompt
        self.llm = _build_llm(temperature)
        self._register(description, capabilities)

    # ── MCP helpers ──────────────────────────────────────────────────

    def _send(self, method: str, params: Dict[str, Any] | None = None) -> MCPResponse:
        msg = MCPMessage(method=method, params=params or {}, source=self.agent_id)
        return self.server.handle(msg)

    def _register(self, description: str, capabilities: List[str]) -> None:
        resp = self._send(
            "agents/register",
            {
                "agent_id": self.agent_id,
                "name": self.name,
                "description": description,
                "capabilities": capabilities,
            },
        )
        assert resp.ok, f"Registration failed: {resp.error}"

    def discover_tools(self, tag: Optional[str] = None) -> List[Dict[str, Any]]:
        resp = self._send("tools/list", {"tag": tag} if tag else {})
        return resp.result

    def call_tool(self, tool_name: str, arguments: Dict[str, Any]) -> Any:
        resp = self._send("tools/call", {"name": tool_name, "arguments": arguments})
        if not resp.ok:
            raise RuntimeError(f"Tool call failed: {resp.error}")
        return resp.result

    def set_context(self, key: str, value: Any) -> None:
        self._send("context/set", {"key": key, "value": value})

    def get_context(self, key: str, default: Any = None) -> Any:
        resp = self._send("context/get", {"key": key, "default": default})
        return resp.result

    def list_context_keys(self) -> List[str]:
        return self._send("context/keys", {}).result

    def list_agents(self) -> List[Dict[str, Any]]:
        return self._send("agents/list", {}).result

    def find_agents(self, capability: str) -> List[Dict[str, Any]]:
        return self._send("agents/find", {"capability": capability}).result

    # ── LLM interaction ──────────────────────────────────────────────

    def invoke_llm(self, user_prompt: str) -> str:
        messages = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=user_prompt),
        ]
        response = self.llm.invoke(messages)
        return response.content


print("MCPAgent base class defined.")

## 8. Specialised Agents

Each agent subclass implements a `run()` method that:
1. Discovers relevant tools via MCP
2. Calls tools as needed
3. Uses the LLM to reason over tool outputs
4. Stores results in the shared context

In [ ]:
class OrchestratorAgent(MCPAgent):
    """Decomposes the user query into sub-tasks and delegates to other agents."""

    def __init__(self, server: MCPServer) -> None:
        super().__init__(
            agent_id="orchestrator",
            name="Orchestrator",
            description="Decomposes user queries and coordinates other agents.",
            capabilities=["planning", "coordination"],
            system_prompt=textwrap.dedent("""\
                You are a planning orchestrator in a multi-agent system.
                Given a user query, break it into exactly three sub-tasks that can be
                handled by the following agents:
                  1. Research Agent – searches the web for relevant information
                  2. Analysis Agent – analyses data, performs calculations, summarises
                  3. Writer Agent – produces a final polished report

                Return your plan as a JSON object with this structure:
                {
                  "research_task": "<description>",
                  "analysis_task": "<description>",
                  "writing_task": "<description>"
                }

                Return ONLY valid JSON, no markdown fences.
            """),
            server=server,
        )

    def plan(self, query: str) -> Dict[str, str]:
        # Dynamic tool discovery – show what is available
        available_tools = self.discover_tools()
        tool_names = [t["name"] for t in available_tools]
        print(f"  [{self.name}] Discovered tools: {tool_names}")

        # Discover peer agents
        agents = self.list_agents()
        agent_names = [a["name"] for a in agents]
        print(f"  [{self.name}] Registered agents: {agent_names}")

        prompt = (
            f"User query: {query}\n\n"
            f"Available tools: {json.dumps(tool_names)}\n"
            f"Available agents: {json.dumps(agent_names)}\n\n"
            f"Create a plan."
        )
        raw = self.invoke_llm(prompt)
        # Robust JSON extraction
        raw = raw.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
        plan = json.loads(raw)
        self.set_context("plan", plan)
        print(f"  [{self.name}] Plan stored in context.")
        return plan


print("OrchestratorAgent defined.")

In [ ]:
class ResearchAgent(MCPAgent):
    """Searches the web using the MCP web_search tool and synthesises findings."""

    def __init__(self, server: MCPServer) -> None:
        super().__init__(
            agent_id="researcher",
            name="Researcher",
            description="Searches the web and gathers relevant information.",
            capabilities=["research", "search"],
            system_prompt=textwrap.dedent("""\
                You are a research agent. Given a research task and search results,
                produce a concise summary of the most relevant findings.
                Include key facts, data points, and source references.
                Return plain text (no JSON).
            """),
            server=server,
        )

    def run(self, task: str) -> str:
        # Discover search-related tools
        search_tools = self.discover_tools(tag="search")
        print(f"  [{self.name}] Search tools found: {[t['name'] for t in search_tools]}")

        # Call the web_search tool via MCP
        search_results = self.call_tool("web_search", {"query": task, "max_results": 4})
        print(f"  [{self.name}] Received search results.")

        # Use LLM to synthesise
        prompt = (
            f"Research task: {task}\n\n"
            f"Search results:\n{search_results}\n\n"
            f"Synthesise the findings into a clear, informative summary."
        )
        synthesis = self.invoke_llm(prompt)

        # Store in shared context
        self.set_context("research_results", synthesis)
        print(f"  [{self.name}] Results stored in context under 'research_results'.")
        return synthesis


print("ResearchAgent defined.")

In [ ]:
class AnalysisAgent(MCPAgent):
    """Reads shared context, performs analysis with calculator, and summarises."""

    def __init__(self, server: MCPServer) -> None:
        super().__init__(
            agent_id="analyst",
            name="Analyst",
            description="Analyses data, performs calculations, and summarises.",
            capabilities=["analysis", "math", "summarisation"],
            system_prompt=textwrap.dedent("""\
                You are an analysis agent. You receive research data and an analysis
                task. Provide a structured analysis including key insights,
                quantitative observations, and a conclusion.
                Return plain text (no JSON).
            """),
            server=server,
        )

    def run(self, task: str) -> str:
        # Discover analysis-related tools
        analysis_tools = self.discover_tools(tag="analysis")
        print(f"  [{self.name}] Analysis tools found: {[t['name'] for t in analysis_tools]}")

        # Read shared context written by ResearchAgent
        research_data = self.get_context("research_results", "No research data available.")
        print(f"  [{self.name}] Retrieved 'research_results' from context store.")

        # Demonstrate calculator tool call
        calc_result = self.call_tool("calculator", {"expression": "42 * 3 + 7"})
        print(f"  [{self.name}] Calculator result: {calc_result}")

        # Demonstrate summariser tool call
        summary_result = self.call_tool(
            "text_summariser",
            {"text": research_data, "max_sentences": 2},
        )
        print(f"  [{self.name}] Summariser result: {summary_result}")

        # LLM-powered analysis
        prompt = (
            f"Analysis task: {task}\n\n"
            f"Research data:\n{research_data}\n\n"
            f"Tool outputs:\n- Calculator: {calc_result}\n- Summary: {summary_result}\n\n"
            f"Provide a thorough analysis."
        )
        analysis = self.invoke_llm(prompt)

        self.set_context("analysis_results", analysis)
        print(f"  [{self.name}] Analysis stored in context under 'analysis_results'.")
        return analysis


print("AnalysisAgent defined.")

In [ ]:
class WriterAgent(MCPAgent):
    """Reads all shared context and produces a final structured report."""

    def __init__(self, server: MCPServer) -> None:
        super().__init__(
            agent_id="writer",
            name="Writer",
            description="Produces polished written reports from analysis.",
            capabilities=["writing", "formatting"],
            system_prompt=textwrap.dedent("""\
                You are a professional technical writer. Given research findings
                and analysis, produce a clear, well-structured report with:
                - An executive summary
                - Key findings
                - Detailed analysis
                - Conclusion and recommendations
                Return the content as plain text sections (no markdown fences).
            """),
            server=server,
        )

    def run(self, task: str) -> str:
        # Discover writing tools
        writing_tools = self.discover_tools(tag="writing")
        print(f"  [{self.name}] Writing tools found: {[t['name'] for t in writing_tools]}")

        # Retrieve all shared context
        context_keys = self.list_context_keys()
        print(f"  [{self.name}] Context keys available: {context_keys}")

        research = self.get_context("research_results", "N/A")
        analysis = self.get_context("analysis_results", "N/A")
        plan = self.get_context("plan", {})

        # LLM-powered writing
        prompt = (
            f"Writing task: {task}\n\n"
            f"Original plan: {json.dumps(plan)}\n\n"
            f"Research findings:\n{research}\n\n"
            f"Analysis:\n{analysis}\n\n"
            f"Write the final report."
        )
        report_text = self.invoke_llm(prompt)

        # Use the format_report tool via MCP
        formatted = self.call_tool(
            "format_report",
            {"title": "Multi-Agent Research Report", "sections": [report_text]},
        )
        print(f"  [{self.name}] Report formatted via MCP tool.")

        self.set_context("final_report", formatted)
        print(f"  [{self.name}] Final report stored in context.")
        return formatted


print("WriterAgent defined.")

## 9. LangGraph Workflow

We wire the agents into a `StateGraph` so the workflow executes as a DAG:

```
  +-------+      +-------------+      +----------+      +---------+      +-------+      +-----+
  | START |----->| Orchestrate |----->| Research |----->| Analyse |----->| Write |----->| END |
  +-------+      +-------------+      +----------+      +---------+      +-------+      +-----+
```

In [ ]:
class WorkflowState(TypedDict):
    query: str
    plan: Dict[str, str]
    research: str
    analysis: str
    report: str
    messages: Annotated[List[str], operator.add]


def build_workflow(server: MCPServer) -> StateGraph:
    orchestrator = OrchestratorAgent(server)
    researcher = ResearchAgent(server)
    analyst = AnalysisAgent(server)
    writer = WriterAgent(server)

    def orchestrate_node(state: WorkflowState) -> dict:
        print("\n=== ORCHESTRATE ===")
        plan = orchestrator.plan(state["query"])
        return {
            "plan": plan,
            "messages": [f"Orchestrator created plan: {json.dumps(plan)}"],
        }

    def research_node(state: WorkflowState) -> dict:
        print("\n=== RESEARCH ===")
        task = state["plan"].get("research_task", state["query"])
        result = researcher.run(task)
        return {
            "research": result,
            "messages": [f"Researcher completed: {task[:80]}..."],
        }

    def analyse_node(state: WorkflowState) -> dict:
        print("\n=== ANALYSE ===")
        task = state["plan"].get("analysis_task", "Analyse the research findings.")
        result = analyst.run(task)
        return {
            "analysis": result,
            "messages": [f"Analyst completed: {task[:80]}..."],
        }

    def write_node(state: WorkflowState) -> dict:
        print("\n=== WRITE ===")
        task = state["plan"].get("writing_task", "Write a comprehensive report.")
        result = writer.run(task)
        return {
            "report": result,
            "messages": [f"Writer completed: {task[:80]}..."],
        }

    graph = StateGraph(WorkflowState)
    graph.add_node("orchestrate", orchestrate_node)
    graph.add_node("research", research_node)
    graph.add_node("analyse", analyse_node)
    graph.add_node("write", write_node)

    graph.add_edge(START, "orchestrate")
    graph.add_edge("orchestrate", "research")
    graph.add_edge("research", "analyse")
    graph.add_edge("analyse", "write")
    graph.add_edge("write", END)

    return graph.compile()


print("build_workflow() defined.")

## 10. Run the Workflow

In [ ]:
workflow = build_workflow(mcp_server)

user_query = (
    "What are the latest advancements in quantum computing, "
    "and how might they impact the cybersecurity landscape over the next five years?"
)

print(f"User query: {user_query}\n")
print("=" * 72)

result = workflow.invoke(
    {
        "query": user_query,
        "plan": {},
        "research": "",
        "analysis": "",
        "report": "",
        "messages": [],
    }
)

print("\n" + "=" * 72)
print("WORKFLOW COMPLETE")
print("=" * 72)

## 11. Inspect Results

### 11.1 Final Report

In [ ]:
print(result["report"])

### 11.2 Workflow Message Trace

In [ ]:
for i, msg in enumerate(result["messages"], 1):
    print(f"  {i}. {msg}")

### 11.3 MCP Context Store (Demonstrates Persistence)

In [ ]:
print("Context store keys:", mcp_server.context_store.keys())
print("\nContext history (last 6 entries):")
for entry in mcp_server.context_store.get_history()[-6:]:
    print(f"  [{entry['timestamp'][:19]}] {entry['source']:>15s} -> {entry['action']}({entry['key']})")

print(f"\nPersisted to: {CONTEXT_PERSIST_PATH}")
if Path(CONTEXT_PERSIST_PATH).exists():
    size_kb = Path(CONTEXT_PERSIST_PATH).stat().st_size / 1024
    print(f"File size: {size_kb:.1f} KB")

### 11.4 Agent Registry

In [ ]:
print("Registered agents:")
for agent in mcp_server.agent_registry.list_agents():
    print(f"  - {agent['name']} (id={agent['agent_id']})")
    print(f"    Capabilities: {agent['capabilities']}")
    print(f"    Status:       {agent['status']}")
    print()

### 11.5 MCP Request Log (Full Audit Trail)

In [ ]:
log = mcp_server.get_request_log()
print(f"Total MCP requests: {len(log)}\n")
for entry in log:
    print(f"  [{entry['ts'][:19]}]  {entry['source']:>15s}  {entry['method']}")

## 12. Demonstrate Dynamic Tool Registration at Runtime

A key MCP feature is that tools can be added **after** the system is running.
Below we register a brand-new tool and immediately use it.

In [ ]:
def sentiment_analyser(text: str) -> str:
    """Very simple rule-based sentiment classifier."""
    positive = sum(1 for w in ["good", "great", "excellent", "positive", "benefit", "advance"]
                   if w in text.lower())
    negative = sum(1 for w in ["bad", "risk", "threat", "danger", "negative", "concern"]
                   if w in text.lower())
    if positive > negative:
        label = "positive"
    elif negative > positive:
        label = "negative"
    else:
        label = "neutral"
    return json.dumps({"sentiment": label, "positive_hits": positive, "negative_hits": negative})


# Register via MCP at runtime
resp = mcp_server.handle(
    MCPMessage(
        method="tools/register",
        params={
            "name": "sentiment_analyser",
            "description": "Classify the sentiment of a text passage.",
            "parameters": {"text": {"type": "string", "description": "Text to analyse"}},
            "handler": sentiment_analyser,
            "tags": ["analysis", "text"],
        },
        source="runtime_admin",
    )
)
print(f"Registration response: {resp.result}")

# Any agent can now discover and use it
print("\nTools now available:")
for t in mcp_server.tool_registry.list_tools():
    print(f"  - {t['name']}: {t['description']}")

# Call the new tool
report_text = mcp_server.context_store.get("final_report", "no report")
sentiment_resp = mcp_server.handle(
    MCPMessage(
        method="tools/call",
        params={"name": "sentiment_analyser", "arguments": {"text": report_text}},
        source="analyst",
    )
)
print(f"\nSentiment of final report: {sentiment_resp.result}")

## 13. Demonstrate Agent Discovery by Capability

Agents can query the MCP server to find peers with specific capabilities.

In [ ]:
for cap in ["research", "analysis", "writing", "planning"]:
    matches = mcp_server.handle(
        MCPMessage(method="agents/find", params={"capability": cap}, source="demo")
    ).result
    names = [m["name"] for m in matches]
    print(f"  Capability '{cap}' -> agents: {names}")

## 14. Demonstrate Context Persistence Across Sessions

Because the `MCPContextStore` writes to disk, data survives across kernel restarts.
Below we create a **fresh** `MCPServer` that reads the previously persisted file.

In [ ]:
fresh_server = MCPServer(persist_path=CONTEXT_PERSIST_PATH)

restored_keys = fresh_server.context_store.keys()
print(f"Restored keys from previous session: {restored_keys}")

restored_plan = fresh_server.context_store.get("plan")
print(f"\nRestored plan: {json.dumps(restored_plan, indent=2)}")

## 15. Clean Up Persistence File

In [ ]:
if Path(CONTEXT_PERSIST_PATH).exists():
    Path(CONTEXT_PERSIST_PATH).unlink()
    print(f"Removed {CONTEXT_PERSIST_PATH}")
else:
    print("Nothing to clean up.")

---

## Summary

This notebook demonstrated the four core MCP capabilities in a multi-agent system:

| Capability | Where demonstrated |
|---|---|
| **Tool Calling** | Every agent calls tools via `tools/call` MCP messages (Sections 8-10) |
| **Dynamic Tool Discovery** | Agents call `tools/list` with optional tag filters before acting (Section 8); a new tool is registered and used at runtime (Section 12) |
| **Contextual Data Sharing & Persistence** | Agents write results to the context store; downstream agents read them; the store is persisted to disk and restored (Sections 11, 14) |
| **Agent Registration** | Every agent registers via `agents/register` on construction; the orchestrator queries `agents/list` and `agents/find` (Sections 8, 13) |

### Class Diagram

```
  +---------------------------+         +---------------------------+
  |        MCPMessage         |         |       MCPResponse         |
  +---------------------------+         +---------------------------+
  | + method : str            |         | + id : str                |
  | + params : Dict           |         | + result : Any            |
  | + id : str                |         | + error : Optional[str]   |
  | + source : str            |         +---------------------------+
  | + timestamp : str         |         | + ok() : bool             |
  +---------------------------+         +---------------------------+

  +-------------------------------+     +-------------------------------+
  |       ToolDefinition          |     |      AgentDescriptor          |
  +-------------------------------+     +-------------------------------+
  | + name : str                  |     | + agent_id : str              |
  | + description : str           |     | + name : str                  |
  | + parameters : Dict           |     | + description : str           |
  | + handler : Callable          |     | + capabilities : List[str]    |
  | + tags : List[str]            |     | + status : str                |
  +-------------------------------+     +-------------------------------+
  | + schema_dict() : Dict        |
  +-------------------------------+

  +-----------------------------------+
  |            MCPServer              |            owns
  +-----------------------------------+   +---------------------+
  | + tool_registry : MCPToolRegistry |-->| MCPToolRegistry     |
  | + agent_registry: MCPAgentRegistry|   +---------------------+
  | + context_store : MCPContextStore |   | + register()        |
  +-----------------------------------+   | + unregister()      |
  | + handle(MCPMessage): MCPResponse |   | + list_tools(tag)   |
  | + get_request_log() : List        |   | + call(name, args)  |
  +-----------------------------------+   +---------------------+
           |              |
           |              |            owns
           |    +--------------------------+
           |    | MCPAgentRegistry         |
           |    +--------------------------+
           |    | + register()             |
           |    | + deregister()           |
           |    | + list_agents()          |
           |    | + get(agent_id)          |
           |    | + find_by_capability()   |
           |    +--------------------------+
           |
           |                           owns
           +---> +---------------------------+
                 | MCPContextStore            |
                 +---------------------------+
                 | + set(key, value, source)  |
                 | + get(key, default)        |
                 | + keys() : List[str]       |
                 | + get_history() : List     |
                 +---------------------------+

  +----------------------------------+
  |           MCPAgent               |     communicates via
  +----------------------------------+   +-------------------+
  | + agent_id : str                 |-->|    MCPServer      |
  | + name : str                     |   +-------------------+
  | + llm : LLM                      |
  +----------------------------------+
  | + discover_tools(tag) : List     |
  | + call_tool(name, args) : Any    |
  | + set_context(key, value)        |
  | + get_context(key) : Any         |
  | + list_context_keys() : List     |
  | + list_agents() : List           |
  | + find_agents(cap) : List        |
  | + invoke_llm(prompt) : str       |
  +----------------------------------+
          ^          ^          ^          ^
          |          |          |          |
          |  inherits|          |          |
  +------------+ +----------+ +-----------+ +----------+
  |Orchestrator| | Research | | Analysis  | | Writer   |
  |   Agent    | |  Agent   | |  Agent    | |  Agent   |
  +------------+ +----------+ +-----------+ +----------+
  | + plan()   | | + run()  | | + run()   | | + run()  |
  +------------+ +----------+ +-----------+ +----------+
```